# Province Pipeline Evaluation (test_images_province)

Notebook นี้ยึดตามโครงจาก pipe_load.ipynb และปรับให้ทดสอบชุดข้อมูล test_images_province โดยใช้ labels.csv คอลัมน์ province_description และ audit_img_rgb.

จุดสำคัญคือมีการตัดค่าใน audit_img_rgb ให้เหลือชื่อไฟล์ .jpg แล้ว map ไปยังไฟล์จริงในโฟลเดอร์ images_by_province.

In [1]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Province classifier deps
import torch
import torch.nn.functional as F
import timm
import torchvision.transforms as T
from PIL import Image, ImageDraw, ImageFont

# ------------------------------
# Fonts (Thai): prefer Tahoma on Windows
def get_tahoma_font(size: int = 20):
    candidates = [
        r"C:\\Windows\\Fonts\\tahoma.ttf",
        r"C:\\Windows\\Fonts\\Tahoma.ttf",
    ]
    for p in candidates:
        try:
            return ImageFont.truetype(p, size=size)
        except Exception:
            pass
    return ImageFont.load_default()

TH_FONT_SMALL = get_tahoma_font(18)

try:
    plt.rcParams['font.family'] = ['Tahoma']
except Exception:
    pass

def draw_text_thai_bgr(
    img_bgr: np.ndarray,
    text: str,
    org_xy: tuple[int, int],
    font: ImageFont.ImageFont,
    fill_bgr=(0, 255, 0),
    stroke_bgr=(0, 0, 0),
    stroke_width: int = 2,
 ):
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    draw = ImageDraw.Draw(pil)
    x, y = org_xy
    fill_rgb = (fill_bgr[2], fill_bgr[1], fill_bgr[0])
    stroke_rgb = (stroke_bgr[2], stroke_bgr[1], stroke_bgr[0])
    draw.text(
        (x, y),
        text,
        font=font,
        fill=fill_rgb,
        stroke_width=stroke_width,
        stroke_fill=stroke_rgb,
    )
    out = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    return out

# ------------------------------
# Paths to weights
WEIGHTS_DIR = Path(r"D:\CodingD\ALPR\weights")
PLATE_DET_W = WEIGHTS_DIR / "plate_detector_best.pt"
PLATE_SPLIT_W = WEIGHTS_DIR / "plate_splitter_best.pt"
PROVINCE_CKPT = WEIGHTS_DIR / "province_classifier_best_new_model.pt"
OCR_CKPT = WEIGHTS_DIR / "upper_ctc_special_best.pt"

# Load models
plate_model = YOLO(PLATE_DET_W)
splitter_model = YOLO(PLATE_SPLIT_W)

SPLIT_COLORS = {0: (255, 0, 0), 1: (0, 128, 255)}

# ------------------------------
# Province classifier helpers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('province_classifier device:', device)

province_tfm = T.Compose([
    T.Resize((32, 128)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def load_province_classifier(ckpt_path: Path):
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Province checkpoint not found: {ckpt_path}")
    ckpt = torch.load(str(ckpt_path), map_location=device)
    model = timm.create_model(
        ckpt['model_name'],
        pretrained=False,
        num_classes=ckpt['num_classes'],
        in_chans=3,
    ).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    idx2label = ckpt.get('idx2label', None)
    if idx2label is None:
        label2idx = ckpt.get('label2idx', {})
        idx2label = {int(v): k for k, v in label2idx.items()}
    else:
        idx2label = {int(k): v for k, v in idx2label.items()}
    return model, idx2label

province_model, idx2label = load_province_classifier(PROVINCE_CKPT)

@torch.no_grad()
def predict_province_from_bgr(bgr_crop: np.ndarray, topk: int = 3):
    if bgr_crop is None or bgr_crop.size == 0:
        return []
    rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    x = province_tfm(img).unsqueeze(0).to(device)
    logits = province_model(x)
    probs = F.softmax(logits, dim=1).squeeze(0)
    k = min(topk, probs.numel())
    vals, idxs = torch.topk(probs, k=k)
    out = []
    for v, i in zip(vals.cpu().tolist(), idxs.cpu().tolist()):
        out.append((idx2label.get(int(i), str(int(i))), float(v)))
    return out

# ------------------------------
# CTC OCR helpers
class CRNN(torch.nn.Module):
    def __init__(self, num_classes: int, img_height: int = 32):
        super().__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, 3, 1, 1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(True),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(64, 128, 3, 1, 1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(True),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(128, 256, 3, 1, 1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(True),
            torch.nn.Conv2d(256, 256, 3, 1, 1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(True),
            torch.nn.MaxPool2d((2, 1), (2, 1)),
            torch.nn.Conv2d(256, 512, 3, 1, 1), torch.nn.BatchNorm2d(512), torch.nn.ReLU(True),
            torch.nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.rnn = torch.nn.LSTM(
            512 * (img_height // 16),
            256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = torch.nn.Linear(256 * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.cnn(x)
        b, c, h, w = feats.size()
        feats = feats.permute(0, 3, 1, 2).contiguous()
        feats = feats.view(b, w, c * h)
        rnn_out, _ = self.rnn(feats)
        logits = self.classifier(rnn_out)
        return logits.permute(1, 0, 2)

def load_ocr_model(ckpt_path: Path):
    if not ckpt_path.exists():
        raise FileNotFoundError(f"OCR checkpoint not found: {ckpt_path}")
    ckpt = torch.load(str(ckpt_path), map_location=device)
    idx_to_char = ckpt.get('idx_to_char', None)
    if idx_to_char is None:
        raise ValueError('idx_to_char not found in OCR checkpoint')
    model = CRNN(num_classes=len(idx_to_char)).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model, idx_to_char

ocr_model, ocr_idx_to_char = load_ocr_model(OCR_CKPT)

ocr_tfm = T.Compose([
    T.Resize((32, 128)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

@torch.no_grad()
def ocr_greedy_decode(logits: torch.Tensor) -> str:
    probs = logits.softmax(2)
    indices = probs.argmax(2).permute(1, 0)
    seq = indices[0].tolist()
    prev = None
    chars = []
    for idx in seq:
        if idx != 0 and idx != prev:
            chars.append(ocr_idx_to_char[idx])
        prev = idx
    return ''.join(chars)

@torch.no_grad()
def predict_text_from_bgr(bgr_crop: np.ndarray):
    if bgr_crop is None or bgr_crop.size == 0:
        return ''
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)
    img = Image.fromarray(gray)
    x = ocr_tfm(img).unsqueeze(0).to(device)
    logits = ocr_model(x)
    return ocr_greedy_decode(logits)

# ------------------------------
def load_bgr(path: Path):
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot load image: {path}")
    return img

print('Models loaded successfully.')

province_classifier device: cuda
Models loaded successfully.


In [2]:
from __future__ import annotations

import pandas as pd
import numpy as np
import re
import time
from pathlib import Path

# ------------------------------
# Config (test_images_province)
IMG_ROOT = Path(r"D:\CodingD\ALPR\pipeline\data\test_images_province_fix")
IMG_DIR = IMG_ROOT / "images_by_province"
LABELS_CSV = IMG_ROOT / "labels.csv"
MAX_SAMPLES = None  # set int for quick test, e.g. 500

# ------------------------------
# Helpers for metrics/path mapping
def levenshtein_seq(a, b) -> int:
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(
                dp[j] + 1,
                dp[j - 1] + 1,
                prev + cost,
            )
            prev = cur
    return dp[m]

def cer(gt: str, pred: str) -> float:
    gt = '' if gt is None else str(gt)
    pred = '' if pred is None else str(pred)
    dist = levenshtein_seq(gt, pred)
    return dist / max(1, len(gt))

def wer(gt: str, pred: str) -> float:
    gt_words = str(gt).split()
    pred_words = str(pred).split()
    dist = levenshtein_seq(gt_words, pred_words)
    return dist / max(1, len(gt_words))

def p95(x: list[float]) -> float:
    return float(np.percentile(np.array(x), 95)) if x else 0.0

def clean_hidden_chars(s: str) -> str:
    # Remove invisible directional marks that can appear in file names
    return re.sub(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]", '', s)

def extract_jpg_filename(raw_path: str) -> str:
    if raw_path is None:
        return ''
    s = clean_hidden_chars(str(raw_path)).strip().strip('"').strip("'")
    s = s.replace('/', '\\')
    tail = s.split('\\')[-1] if '\\' in s else s

    # Keep only the file name up to .jpg/.jpeg (even if there is extra trailing text)
    m = re.search(r"(.+?\.(?:jpg|jpeg))", tail, flags=re.IGNORECASE)
    if m:
        name = m.group(1)
        return re.sub(r"_CTX(?=\.(?:jpg|jpeg)$)", '', name, flags=re.IGNORECASE)

    # Fallback for unexpected formats
    m2 = re.search(r"([^\\/:*?\"<>|\r\n]+?\.(?:jpg|jpeg))", s, flags=re.IGNORECASE)
    if m2:
        name = m2.group(1)
        return re.sub(r"_CTX(?=\.(?:jpg|jpeg)$)", '', name, flags=re.IGNORECASE)

    return tail

def build_image_index(root_dir: Path) -> dict[str, list[Path]]:
    index: dict[str, list[Path]] = {}
    for img_path in root_dir.rglob('*.jpg'):
        index.setdefault(img_path.name, []).append(img_path)
    for img_path in root_dir.rglob('*.jpeg'):
        index.setdefault(img_path.name, []).append(img_path)
    return index

# ------------------------------
# Load labels
if not LABELS_CSV.exists():
    raise FileNotFoundError(f'labels.csv not found: {LABELS_CSV}')
if not IMG_DIR.exists():
    raise FileNotFoundError(f'Image directory not found: {IMG_DIR}')

df = pd.read_csv(LABELS_CSV, dtype=str).fillna('')
required_cols = {'province_description', 'audit_img_rgb'}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f'Missing columns in labels.csv: {missing_cols}')

# Build filename and optional gt plate text
df['filename'] = df['audit_img_rgb'].apply(extract_jpg_filename)
if {'plate1', 'plate2'}.issubset(df.columns):
    df['gt_text'] = (df['plate1'].astype(str).str.strip() + df['plate2'].astype(str).str.strip()).str.replace(' ', '', regex=False)
else:
    df['gt_text'] = ''

print('Building image index...')
img_index = build_image_index(IMG_DIR)
print(f'Indexed {sum(len(v) for v in img_index.values())} images, {len(img_index)} unique names')

def resolve_image_path(row) -> Path | None:
    province = str(row['province_description']).strip()
    filename = str(row['filename']).strip()
    if not filename:
        return None

    # Preferred path: images_by_province/<province_description>/<filename>
    direct_path = IMG_DIR / province / filename
    if direct_path.exists():
        return direct_path

    # Fallback by filename index
    candidates = img_index.get(filename, [])
    if not candidates:
        return None

    if province:
        for p in candidates:
            if p.parent.name == province:
                return p
    return candidates[0]

df['image_path'] = df.apply(resolve_image_path, axis=1)
matched = df['image_path'].notna().sum()
print(f'Matched rows: {matched}/{len(df)}')

if MAX_SAMPLES is not None and matched > MAX_SAMPLES:
    df = df[df['image_path'].notna()].sample(MAX_SAMPLES, random_state=42).reset_index(drop=True)
else:
    df = df[df['image_path'].notna()].reset_index(drop=True)

if len(df) == 0:
    raise RuntimeError('No matched rows. Please verify labels.csv and image files in images_by_province.')

# ------------------------------
# Evaluation loop
plate_found = 0
text_box_found = 0
prov_box_found = 0

text_acc = 0
text_gt_count = 0
prov_acc = 0
cer_list = []
wer_list = []

t_plate = []
t_split = []
t_ocr = []
t_prov = []
t_total = []

y_true_prov = []
y_pred_prov = []

for _, row in df.iterrows():
    gt_prov = str(row['province_description']).strip()
    gt_text = str(row['gt_text']).strip()
    img_path = Path(row['image_path'])

    frame = load_bgr(img_path)
    t0 = time.perf_counter()

    # Step 1: plate detector
    t1 = time.perf_counter()
    plate_results = plate_model.predict(frame, conf=0.25, iou=0.7, imgsz=1280, verbose=False)[0]
    t_plate.append(time.perf_counter() - t1)

    plate_boxes = plate_results.boxes.xyxy.cpu().numpy() if plate_results.boxes is not None else []
    plate_confs = plate_results.boxes.conf.cpu().numpy() if plate_results.boxes is not None else []

    if len(plate_boxes) == 0:
        t_ocr.append(0.0)
        t_prov.append(0.0)
        t_split.append(0.0)
        t_total.append(time.perf_counter() - t0)
        if gt_text:
            text_gt_count += 1
            cer_list.append(cer(gt_text, ''))
            wer_list.append(wer(gt_text, ''))
        y_true_prov.append(gt_prov)
        y_pred_prov.append('<MISS>')
        continue

    plate_found += 1
    best_idx = int(np.argmax(plate_confs))
    x1, y1, x2, y2 = map(int, plate_boxes[best_idx])
    plate_crop = frame[y1:y2, x1:x2]

    if plate_crop.size == 0:
        t_ocr.append(0.0)
        t_prov.append(0.0)
        t_split.append(0.0)
        t_total.append(time.perf_counter() - t0)
        if gt_text:
            text_gt_count += 1
            cer_list.append(cer(gt_text, ''))
            wer_list.append(wer(gt_text, ''))
        y_true_prov.append(gt_prov)
        y_pred_prov.append('<MISS>')
        continue

    # Step 2: splitter
    t2 = time.perf_counter()
    split_res = splitter_model.predict(plate_crop, conf=0.25, iou=0.6, imgsz=640, verbose=False)[0]
    t_split.append(time.perf_counter() - t2)

    split_boxes = split_res.boxes.xyxy.cpu().numpy() if split_res.boxes is not None else []
    split_cls = split_res.boxes.cls.cpu().numpy().astype(int) if split_res.boxes is not None else []
    split_confs = split_res.boxes.conf.cpu().numpy() if split_res.boxes is not None else []

    text_pred = ''
    prov_pred = '<MISS>'

    # license_text class = 0
    text_idxs = [i for i, c in enumerate(split_cls) if int(c) == 0]
    if text_idxs:
        text_box_found += 1
        best_t = max(text_idxs, key=lambda i: split_confs[i])
        x1t, y1t, x2t, y2t = map(int, split_boxes[best_t])
        text_crop = plate_crop[y1t:y2t, x1t:x2t]
        t3 = time.perf_counter()
        text_pred = predict_text_from_bgr(text_crop)
        t_ocr.append(time.perf_counter() - t3)
    else:
        t_ocr.append(0.0)

    # province class = 1
    prov_idxs = [i for i, c in enumerate(split_cls) if int(c) == 1]
    if prov_idxs:
        prov_box_found += 1
        best_p = max(prov_idxs, key=lambda i: split_confs[i])
        x1p, y1p, x2p, y2p = map(int, split_boxes[best_p])
        prov_crop = plate_crop[y1p:y2p, x1p:x2p]
        t4 = time.perf_counter()
        top = predict_province_from_bgr(prov_crop, topk=1)
        t_prov.append(time.perf_counter() - t4)
        if top:
            prov_pred = str(top[0][0])
    else:
        t_prov.append(0.0)

    # Metrics
    if gt_text:
        text_gt_count += 1
        if text_pred == gt_text:
            text_acc += 1
        cer_list.append(cer(gt_text, text_pred))
        wer_list.append(wer(gt_text, text_pred))

    if prov_pred == gt_prov:
        prov_acc += 1

    y_true_prov.append(gt_prov)
    y_pred_prov.append(prov_pred)
    t_total.append(time.perf_counter() - t0)

# ------------------------------
# Summary
n = len(df)
plate_rate = plate_found / max(1, n)
text_box_rate = text_box_found / max(1, plate_found)
prov_box_rate = prov_box_found / max(1, plate_found)
prov_acc_all = prov_acc / max(1, n)

print('=== Dataset ===')
print(f'Evaluated rows: {n}')
print('=== Detection / Split rates ===')
print(f'Plate detection rate: {plate_rate*100:.2f}% ({plate_found}/{n})')
print(f'Text box found rate: {text_box_rate*100:.2f}% ({text_box_found}/{max(1, plate_found)})')
print(f'Province box found rate: {prov_box_rate*100:.2f}% ({prov_box_found}/{max(1, plate_found)})')

print('=== Accuracy ===')
print(f'Province exact match acc: {prov_acc_all*100:.2f}%')
if text_gt_count > 0:
    text_acc_all = text_acc / max(1, text_gt_count)
    print(f'Plate text exact match acc: {text_acc_all*100:.2f}% (on {text_gt_count} rows with gt text)')
    print(f'CER (avg): {float(np.mean(cer_list)):.4f}')
    print(f'WER (avg): {float(np.mean(wer_list)):.4f}')
else:
    print('No text ground truth available (plate1/plate2) -> skip OCR text metrics.')

print('=== Timing (ms) ===')
def ms(x):
    return [v * 1000 for v in x]
if t_plate:
    print(f'Plate detector: mean={np.mean(ms(t_plate)):.1f} p95={p95(ms(t_plate)):.1f}')
if t_split:
    print(f'Splitter: mean={np.mean(ms(t_split)):.1f} p95={p95(ms(t_split)):.1f}')
if t_prov:
    print(f'Province cls: mean={np.mean(ms(t_prov)):.1f} p95={p95(ms(t_prov)):.1f}')
if t_ocr:
    print(f'OCR: mean={np.mean(ms(t_ocr)):.1f} p95={p95(ms(t_ocr)):.1f}')
if t_total:
    print(f'End-to-end: mean={np.mean(ms(t_total)):.1f} p95={p95(ms(t_total)):.1f} | FPS={1.0/max(1e-9, np.mean(t_total)):.2f}')

print('=== Confusion Matrix (province) ===')
cm = pd.crosstab(
    pd.Series(y_true_prov, name='true'),
    pd.Series(y_pred_prov, name='pred'),
    dropna=False,
 )
print(cm)

print('=== Top Confusions (province) ===')
conf = cm.copy()
for lbl in cm.index:
    if lbl in conf.columns:
        conf.loc[lbl, lbl] = 0
conf_flat = conf.stack().sort_values(ascending=False)
print(conf_flat.head(20))

Building image index...
Indexed 1128 images, 1128 unique names
Matched rows: 1111/1555
=== Dataset ===
Evaluated rows: 1111
=== Detection / Split rates ===
Plate detection rate: 95.14% (1057/1111)
Text box found rate: 100.00% (1057/1057)
Province box found rate: 100.00% (1057/1057)
=== Accuracy ===
Province exact match acc: 72.73%
Plate text exact match acc: 86.05% (on 1111 rows with gt text)
CER (avg): 0.0789
WER (avg): 0.1395
=== Timing (ms) ===
Plate detector: mean=17.3 p95=18.1
Splitter: mean=6.8 p95=8.4
Province cls: mean=3.6 p95=4.5
OCR: mean=2.1 p95=2.9
End-to-end: mean=30.8 p95=33.6 | FPS=32.47
=== Confusion Matrix (province) ===
pred           <MISS>  กระบี่  กรุงเทพมหานคร  กาญจนบุรี  กาฬสินธุ์  กำแพงเพชร  \
true                                                                            
กระบี่              1       2              0          0          0          0   
กรุงเทพมหานคร       0       0             18          0          0          0   
กาญจนบุรี           1       0 